In [2]:
!pip install google-cloud-spanner google-cloud-aiplatform pydantic

In [11]:
# Configuration
PROJECT_ID = "bq-project-402513"
INSTANCE_ID = "graph"
DATABASE_ID = "supportgraph"
GRAPH_NAME = "SupportContextGraph"
GEMINI_MODEL = "gemini-2.5-flash"


# Set these before initializing your Runner
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1" # or your preferred region
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

In [16]:
import vertexai
from vertexai.generative_models import GenerativeModel, Tool, FunctionDeclaration, Content, Part
from google.cloud import spanner
from google.cloud.spanner_admin_database_v1.types import common

vertexai.init(project=PROJECT_ID, location="us-central1")
spanner_client = spanner.Client()
instance = spanner_client.instance(INSTANCE_ID)
database = instance.database(DATABASE_ID)

# --- 2. Define the Contextual Policy Function ---
def check_retention_history(customer_id: str):
    """
    Searches the Spanner Context Graph and returns a structured 
    Intelligence Report for the AI Agent.
    """
    gql_query = f"""
    GRAPH SupportContextGraph
    MATCH (c:Customers {{customer_id: '{customer_id}'}})<-[:AboutCustomer]-(d:Decisions)
    MATCH (d)-[:ResultedIn]->(o:Outcomes)
    RETURN 
      d.timestamp AS Date,
      d.amount AS Past_Amount,
      d.reasoning_text AS Past_Reasoning,
      o.result AS Past_Outcome
    ORDER BY d.timestamp DESC
    """
    
    try:
        with database.snapshot() as snapshot:
            results = snapshot.execute_sql(gql_query)
            rows = list(results)
            
            # --- START STRUCTURED REPORT ---
            report = "\n" + "="*50 + "\n"
            report += "🔍 CONTEXT GRAPH INTELLIGENCE REPORT\n"
            report += "="*50 + "\n"

            if not rows:
                report += "🟢 STATUS: NO PRIOR CONFLICTS\n"
                report += "ADVICE: Proceed with standard tier-based discount policy.\n"
                return report

            # Logic to find the "Warning" path
            found_warning = False
            for row in rows:
                if row[1] >= 0.40 and row[3] == 'Churned':
                    found_warning = True
                    report += "⚠️ PRECEDENT FOUND: HIGH-RISK FAILURE\n"
                    report += f"   • Date: {row[0].date()}\n"
                    report += f"   • Action: {row[1]*100}% Discount\n"
                    report += f"   • Reasoning: \"{row[2]}\"\n"
                    report += f"   • Outcome: ❌ {row[3].upper()}\n\n"
                    
                    report += "🛡️ POLICY GUARDRAIL (POL-99):\n"
                    report += "   STATUS: BLOCKING REPETITIVE FAILED TACTIC\n"
                    report += "   STRATEGY: Pivot to Value-Based Service (Technical Workshop).\n"
                    break
            
            if not found_warning:
                report += "✅ STATUS: POSITIVE HISTORICAL ALIGNMENT\n"
                report += "ADVICE: Historical data supports current retention strategy.\n"

            report += "="*50 + "\n"
            return report

    except Exception as e:
        return f"❌ Error querying Context Graph: {str(e)}"

# --- 3. Wrap in Vertex AI Tooling ---
retention_tool = Tool(
    function_declarations=[
        FunctionDeclaration(
            name="check_retention_history",
            description="Lookup historical decisions and outcomes from the Context Graph.",
            parameters={
                "type": "object",
                "properties": {
                    "customer_id": {"type": "string", "description": "The ID of the customer"}
                },
                "required": ["customer_id"]
            },
        )
    ]
)


In [17]:

# --- 4. The Demo Execution ---
model = GenerativeModel(GEMINI_MODEL, tools=[retention_tool])
chat = model.start_chat()

prompt = "Customer CUST-001 is threatening to churn. Should I give them a 50% discount to keep them?"

print(f"User Request: {prompt}\n")


User Request: Customer CUST-001 is threatening to churn. Should I give them a 50% discount to keep them?



In [18]:

# Initial call to the model
response = chat.send_message(prompt)

# Handle the Function Call (The 'Thinking' phase)
if response.candidates[0].function_calls:
    function_call = response.candidates[0].function_calls[0]
    if function_call.name == "check_retention_history":
        # Execute the real Spanner Graph query
        context_result = check_retention_history(function_call.args["customer_id"])
        print(f"--- Context Graph Lookup Result ---\n{context_result}\n")
        
        # Send result back to Gemini to get the final "Governed" answer
        final_response = chat.send_message(
            Part.from_function_response(
                name="check_retention_history",
                response={"content": context_result}
            )
        )
        print(f"Final Agent Decision:\n{final_response.text}")

Created multiplexed session.
Created multiplexed session.
INFO:projects/bq-project-402513/instances/graph/databases/supportgraph:Created multiplexed session.


--- Context Graph Lookup Result ---

🔍 CONTEXT GRAPH INTELLIGENCE REPORT
⚠️ PRECEDENT FOUND: HIGH-RISK FAILURE
   • Date: 2025-03-15
   • Action: 50.0% Discount
   • Reasoning: "Customer threatened to leave for a cheaper competitor. Match price to save account."
   • Outcome: ❌ CHURNED

🛡️ POLICY GUARDRAIL (POL-99):
   STATUS: BLOCKING REPETITIVE FAILED TACTIC
   STRATEGY: Pivot to Value-Based Service (Technical Workshop).


Final Agent Decision:
Based on the retention history, giving CUST-001 a 50% discount on 2025-03-15 was ineffective, as the customer churned anyway. Policy Guardrail (POL-99) advises against repeating this failed tactic. Instead, you should pivot to a value-based service, such as a Technical Workshop.
